# U1.L2 — Multi-scale flows on images

**Session U1.L2** (lab, joint, *elastic*). Companion to the U1.L2 lecture notes, which
explain every step here at length; this notebook is the part you execute.

By the end you own three things:

1. the **`bpd-protocol`**: a complete convention — observation model, dequantization,
   preprocessing, evaluation — under which a "bits per dimension" number on MNIST means
   one thing, and under which later likelihood numbers of this course are comparable;
2. the **multi-scale bookkeeping**, executable: a squeeze has log-determinant exactly
   zero, a factor-out is a partition and not a transform, and a per-level table whose
   rows add up to `log_prob`;
3. a **course-internal comparison** of two models under one protocol, and the list of
   reasons why a number copied from a paper is not on the same axis.

### How to read this notebook

Cells marked **TODO** are yours. Each TODO is followed by a **Checkpoint** cell that
either passes or tells you what is wrong. Everything else is given code: run it, read
it, move on. If you are working through this on your own — this session is the first
one cut when the calendar slips, and the notebook is written for that case — the U1.L2
notes carry the reasoning behind each block, and the solutions notebook,
`lab-u1l2-solutions.ipynb`, which the instructor releases, carries the filled-in cells.

**Everything runs on CPU.** The live training of block 2 takes about 23 s;
evaluating the full 10,000-image test set through the reference checkpoint takes about
14 s. Both numbers were measured on a laptop core.

## Block 0 — Why images change the problem

U1.L1 trained a coupling flow on points in the plane by exact maximum likelihood, and
nothing about the algebra depended on $d = 2$. Two things break when the data become
images, and neither is the change-of-variables theorem.

1. **MNIST is observed as discrete 8-bit values.** Each pixel is an integer
   $k \in \{0, \dots, 255\}$; the test set contains exactly 256 distinct
   values, and 81% of all its pixels are exactly $0$. A continuous density
   evaluated at an integer is not a probability mass, and maximizing a continuous
   density on atoms is ill-posed: the density can grow without bound by concentrating
   spikes at the observed integers (the notes draw this). Block 1 fixes the observation
   model before any likelihood is computed.
2. **$d = 784$.** A fully connected conditioner on a 784-vector is no longer a toy,
   and the Jacobian bookkeeping of a deep stack has to be organized. Block 2 is that
   organization: squeeze, levels, factor-out.

### Given code

Four modules under `common/` arrive as **given code**; the bootcamp is provenance, not a
prerequisite, and the contract of each is written here.

| Module | Built in | What this lab uses |
|---|---|---|
| `common/train.py` | U0.L1, extended in U0.L2 | `set_seed(seed)`; `Trainer(...)` with `.fit(train_loader, val_loader, epochs)`, as in U1.L1 |
| `common/eval.py` | U0.L4 | **`bpd(log_prob_nats_per_example, num_dims)`** only. Its docstring reserves the dequantization term for "when the course first uses BPD, in U1" — that is today |
| `common/coupling.py` | U1.L1 | the affine algebra and `HEAD_INIT_SCALE`; nothing new |
| `common/multiscale_flow.py` | **today** (block 2) | `Logit`, `Squeeze2d`, `ImageAffineCoupling`, `MultiscaleFlow` |

### The data-loader rule

Load MNIST **as it is stored**: `uint8`, values in $\{0, \dots, 255\}$. Keep it that way
until the dequantization cell of block 1. In particular, do **not** let
`torchvision.transforms.ToTensor()` in: it divides by $255$ (not $256$), returns floats,
and hides the factor whose logarithm becomes the $+8$ of block 1. The cell below loads
the raw tensor and checks that it is what we say it is.

In [ ]:
import sys
from pathlib import Path

# The repository root, so `import common.train` works from labs/.
ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(ROOT))

import hashlib
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

from common.eval import bpd
from common.train import Trainer, set_seed

torch.set_num_threads(4)
set_seed(0)
DEVICE = "cpu"          # deliberate: the lab is CPU-sufficient throughout
DATA = ROOT / "scripts" / "python" / "data"


def load_raw(train):
    """MNIST as stored: uint8 in {0,...,255}, shape (N, 1, 28, 28). No transform."""
    ds = datasets.MNIST(str(DATA), train=train, download=False)
    return ds.data.clone().unsqueeze(1)


k_test = load_raw(False)
print("torch", torch.__version__)
print("test set:", tuple(k_test.shape), k_test.dtype, "min", int(k_test.min()), "max", int(k_test.max()))

**Checkpoint 0** — the tensor is integer-valued, 8-bit, and untouched; and here is what
`ToTensor()` would have done to it.

In [ ]:
from torchvision import transforms

assert k_test.dtype == torch.uint8 and k_test.shape == (10_000, 1, 28, 28)
assert int(k_test.min()) == 0 and int(k_test.max()) == 255
distinct = int((torch.bincount(k_test.flatten().long(), minlength=256) > 0).sum())
pil = datasets.MNIST(str(DATA), train=False, download=False)[0][0]      # a PIL image
tt = transforms.ToTensor()(pil)
print(f"distinct pixel values in the test set: {distinct}")
print(f"ToTensor(): dtype {tt.dtype}, max {float(tt.max()):.6f} = 255/255 -> the factor is 255, and it is gone")
print("Checkpoint 0 passed: raw uint8 preserved until the dequantization cell.")

## Block 1 — Dequantization and bits per dimension

*Produces the artifact `bpd-protocol`.*

### 1.1 Uniform dequantization, and which way the inequality goes

For a raw pixel vector $k \in \{0,\dots,255\}^D$ and $u \sim \mathrm{Unif}([0,1)^D)$,
define
$$
x = \frac{k + u}{256} \in [0, 1)^D .
$$

If $p_X$ is the continuous density the flow models on $[0,1]^D$, the discrete model it
induces is the mass of the cell of $k$,
$$
P_\theta(k) = \int_{[k/256,\,(k+1)/256)} p_X(x)\,dx
= \frac{1}{256^{D}}\;\mathbb{E}_{u \sim \mathrm{Unif}([0,1)^D)}\!\left[p_X\!\left(\tfrac{k+u}{256}\right)\right],
$$
and Jensen's inequality ($\log$ is concave) gives
$$
\log P_\theta(k)
= \log \mathbb{E}_{u}\!\left[p_X\!\left(\tfrac{k+u}{256}\right)\right] - D\log 256
\;\ge\;
\mathbb{E}_{u}\!\left[\log p_X\!\left(\tfrac{k+u}{256}\right)\right] - D\log 256 .
$$

Read the direction carefully. The quantity a flow is trained on and evaluated with —
the expectation of the log-density on dequantized data — is a **lower bound on the
discrete log-mass**, equivalently an **upper bound on the discrete negative
log-likelihood, in expectation over $u$**. A finite Monte Carlo estimate of that
expectation (one $u$ per image, as below) fluctuates around the bound value; do not call
a one-sample realization an exact bound. Block 4 measures the fluctuation.

The **fixed bank**: for course comparisons, one draw of $u$ per test image, from a
generator seeded with $2026$, on CPU, in test-set order. Same bank, same numbers; the
SHA-256 of the bank the reference run used is recorded, so that a different `torch`
build is detected rather than silently accepted.

In [ ]:
DEQ_SEED, LAM, D = 2026, 1e-6, 784


def deq_bank(n, seed=DEQ_SEED):
    """u ~ Unif[0,1)^D, one draw per image, CPU, seeded: the frozen bank."""
    g = torch.Generator().manual_seed(seed)
    return torch.rand(n, 1, 28, 28, generator=g)


def dequantize(k, u):
    return (k.float() + u) / 256.0


u_test = deq_bank(k_test.shape[0])
x_test = dequantize(k_test, u_test)
bank_sha = hashlib.sha256(u_test.numpy().tobytes()).hexdigest()
print("bank sha256:", bank_sha[:12], "...")

**Checkpoint 1.1** — $x \in [0, 1)$, the noise is recoverable from $x$ and $k$, and the
bank is the one the reference run used (`ead91b278e9c…`).

In [ ]:
assert float(x_test.min()) >= 0.0 and float(x_test.max()) < 1.0
assert torch.allclose(x_test * 256 - k_test.float(), u_test, atol=1e-5)
assert bank_sha == "ead91b278e9c38450c95d2f251ab38708b89362a3c1d4ef3886419adfac6c4f5", "a different noise bank: your torch build draws differently"
print("Checkpoint 1.1 passed: x in [0,1), 256 applied once, bank identical to the reference.")

### 1.2 Logit preprocessing is a flow layer — TODO 1.2

A flow on $[0,1]^D$ has to fight its own boundaries; the standard remedy is to model
the density in logit space. The course-stable transform is
$$
a = \lambda + (1 - 2\lambda)\,x, \qquad y = T_\lambda(x) = \operatorname{logit}(a) = \log a - \log(1 - a),
\qquad \lambda = 10^{-6},
$$
applied per pixel. It is a bijection of $[0,1]^D$ onto a box in $\mathbb{R}^D$, and its
Jacobian is diagonal, so
$$
\log\lvert\det J_{T_\lambda}(x)\rvert
= \sum_{j=1}^{D}\Big[\log(1-2\lambda) - \log a_j - \log(1 - a_j)\Big].
$$

If the flow models $p_Y(y)$, then by the change-of-variables theorem (U1.T1)
$$
\log p_X(x) = \log p_Y\big(T_\lambda(x)\big) + \log\lvert\det J_{T_\lambda}(x)\rvert .
$$

**The preprocessing contribution is never hidden or dropped.** Write the per-example
log-Jacobian by hand, then compare it with `Logit.inverse` from the scaffold, whose
`inverse` is the density direction $x \to y$ (course contract: `forward` is $\psi$,
source to data, so the *logit* is an `inverse`).

In [ ]:
from common.multiscale_flow import Logit


def logit_logdet(x, lam=LAM):
    """(B, 1, 28, 28) in [0,1] -> per-example log|det J_T(x)|, shape (B,)."""
    # TODO 1.2: a = lam + (1 - 2 lam) x; sum over pixels of log(1-2lam) - log a - log(1-a).
    raise NotImplementedError

**Checkpoint 1.2** — your sum against the scaffold's, in float64; and the generative
direction undoes it.

In [ ]:
logit = Logit(LAM)
x64 = x_test[:64].double()
y64, ld_scaffold = logit.inverse(x64)
ld_mine = logit_logdet(x64)
x_back, ld_fwd = logit.forward(y64)
print(f"max |mine - scaffold| = {float((ld_mine - ld_scaffold).abs().max()):.1e}   "
      f"mean log-Jacobian per image = {float(ld_mine.mean()):.1f} nats")
assert float((ld_mine - ld_scaffold).abs().max()) < 1e-8
assert float((x_back - x64).abs().max()) < 1e-12 and float((ld_fwd + ld_scaffold).abs().max()) < 1e-8
print("Checkpoint 1.2 passed.")

### 1.3 Bits per dimension, and the $+8$ — TODO 1.3

The `eval-harness` contract, frozen in U0.L4:

```
bpd(log_prob_nats_per_example, num_dims)
    = -mean(log_prob_nats_per_example) / (num_dims * log 2)
```

It returns the **continuous-density term only**. With $\log p_X$ in nats on
$[0,1)^{784}$, the dequantized discrete objective is
$$
\widehat{\mathrm{bpd}}_{\mathrm{deq}} = \mathrm{bpd}_{\mathrm{cont}} + 8,
\qquad\text{because}\qquad \frac{\log 256}{\log 2} = 8 :
$$
the $-D\log 256$ of the inequality above, divided by $D \log 2$. The $+8$ belongs to
`bpd-protocol`, not to the generic harness. Two sanity facts to keep: a **uniform**
density on $[0,1)^{784}$ has $\log p_X = 0$ and reports exactly $8$ bits per dimension —
the raw 8-bit code; and a model that concentrates has $\log p_X > 0$, so
$\mathrm{bpd}_{\mathrm{cont}}$ is **negative** and $\widehat{\mathrm{bpd}}_{\mathrm{deq}}$
lands below $8$.

::: contract
**Artifact `bpd-protocol` — frozen, for MNIST course comparisons**

- raw 8-bit **test** split; $D = 784$;
- full test set $N = 10{,}000$, unless a session explicitly labels a CPU-sized
  diagnostic subset;
- one fixed uniform dequantization noise bank per image (`K_deq = 1`, seed $2026$);
- $\lambda = 10^{-6}$ for the logit transform;
- the logit Jacobian is included in $\log p_X$ **before** calling `eval_harness.bpd`;
- reported number: `eval_harness.bpd(log_prob_x, 784) + 8`;
- every report logs: dataset and split, $N$, dequantization seed and `K_deq`, $\lambda$,
  checkpoint hash and config.

Publication-grade variants — several $u$ draws per image, or a learned (variational)
dequantization distribution — are named here and are outside this lab.
:::

Write `bpd_protocol`, and the manual arithmetic beside it.

In [ ]:
def bpd_protocol(log_prob_x):
    """The reported number: eval_harness.bpd(log_prob_x, 784) + 8."""
    # TODO 1.3a
    raise NotImplementedError


def bpd_manual(log_prob_x):
    """The same number by hand: -mean / (784 log 2) + log 256 / log 2."""
    # TODO 1.3b
    raise NotImplementedError

**Checkpoint 1.3** — the two agree to float precision, the uniform density reports
exactly $8$, and a density that is $e^{784}$ at every point reports $8 - 1/\log 2$.

In [ ]:
zeros = np.zeros(10)
assert abs(bpd_protocol(zeros) - 8.0) < 1e-12 and abs(bpd_manual(zeros) - 8.0) < 1e-12
big = np.full(10, 784.0)
assert abs(bpd_protocol(big) - (8.0 - 1 / math.log(2.0))) < 1e-9
probe = np.random.default_rng(0).normal(-3000.0, 50.0, size=100)
assert abs(bpd_protocol(probe) - bpd_manual(probe)) < 1e-12
print(f"uniform density: {bpd_protocol(zeros):.1f} bits/dim (the raw code); probe: "
      f"harness {bpd_protocol(probe):.6f}  manual {bpd_manual(probe):.6f}")
print("Checkpoint 1.3 passed.")

## Block 2 — The multi-scale shell

*Uses the artifact `coupling-layer-scratch` (U1.L1); the picture is U1.T2 §2.*

`common/multiscale_flow.py` exposes three pieces you can inspect independently. Every
one of them has the U1.L1 contract — `forward(z) -> (x, logdet_fwd)` is $\psi$, source
to data; `inverse(x) -> (z, logdet_inv)` is the density direction; the two log-dets
cancel per example.

```
Squeeze2d(factor=2)                 # a reshape; exact inverse; log-det 0
ImageAffineCoupling(C, mask, ...)   # U1.L1's layer, mask over CHANNELS, conv conditioner
MultiscaleFlow(in_shape, levels,    # per level: squeeze -> `steps` couplings
               steps, hidden)       #   -> factor out half the channels
```

Two facts the code makes executable, and which the notes' Board 2 draws:

- **Squeeze is a permutation.** It rearranges the four sub-positions of every
  $2 \times 2$ block into four channels: $1 \times 28 \times 28 \to 4 \times 14 \times 14$.
  Nothing is scaled; the log-determinant is exactly $0$. And a *channel* mask after a
  squeeze is a *spatial* mask on the original image: reading channels $\{0, 1\}$ is
  reading the top row of every block.
- **Factor-out is a partition, not a transform.** At the end of a level, half the
  channels go to the latent collection *as they are* ($z_1$), the other half continue.
  No Jacobian term. What $z_1$ contributes is $\log p_0(z_1)$, added at the end:
$$
\log p_Y(y) = \sum_{l=1}^{L} \log p_0(z_l) + \sum_{\text{couplings}} \log\lvert\det J\rvert,
\qquad z = (z_1, \dots, z_L),\ \ \textstyle\sum_l \dim z_l = 784 .
$$

Names follow the density direction, as the papers draw it: `Squeeze2d.inverse`
*squeezes* and `MultiscaleFlow.inverse` *factors out*; the generative `forward` runs
every arrow backwards.

### TODO 2.1 — Squeeze: exact round trip, zero log-det, and what the channels are

In [ ]:
from common.multiscale_flow import (ImageAffineCoupling, MultiscaleFlow, Squeeze2d,
                                    count_params, log_p0_flat)

sq = Squeeze2d(2)
y8 = logit.inverse(x_test[:8])[0]                 # (8, 1, 28, 28) in logit space
# TODO 2.1: z, ld = sq.inverse(y8); y_back, ld_back = sq.forward(z).
#   Assert: z.shape == (8, 4, 14, 14); torch.equal(y_back, y8); both log-dets are
#   exactly 0.0; and channel 0 of z is y8[:, 0, 0::2, 0::2] (top-left of every block).
raise NotImplementedError

### TODO 2.2 — Wire a two-level MNIST model, and the density of $x$

Build the live model — two levels, four coupling steps per level, conditioners of width
48 — and compose the density of the *data* from the two pieces of block 1:
$\log p_X(x) = \log p_Y(T_\lambda(x)) + \log\lvert\det J_{T_\lambda}(x)\rvert$. The
scaffold's `log_prob(y)` is $\log p_Y$; the Jacobian is yours to add, and it stays
yours in block 4 and in PS1.4(d).

In [ ]:
set_seed(0)
flow_live = MultiscaleFlow((1, 28, 28), levels=2, steps=4, hidden=48)
print("latent shapes:", flow_live.latent_shapes, " params:", count_params(flow_live))


def log_prob_x(flow, k, u):
    """Raw uint8 k and noise u -> log p_X(x) per example, Jacobian INCLUDED."""
    # TODO 2.2: x = dequantize(k, u); (y, ld_T) = logit.inverse(x); return flow.log_prob(y) + ld_T
    raise NotImplementedError

**Checkpoint 2.2** — the latent has $392 + 392 = 784$ entries; `log_prob_x` has shape
`(B,)`; and the whole stack is a bijection in float64 (both round trips, cancellation).

In [ ]:
assert flow_live.latent_shapes == [(2, 14, 14), (8, 7, 7)]
assert sum(math.prod(s) for s in flow_live.latent_shapes) == 784
with torch.no_grad():
    lp = log_prob_x(flow_live, k_test[:16], u_test[:16])
    assert lp.shape == (16,) and bool(torch.isfinite(lp).all())
    f64 = MultiscaleFlow((1, 28, 28), levels=2, steps=4, hidden=48).double()
    f64.load_state_dict(flow_live.state_dict())
    y16 = logit.inverse(x_test[:16].double())[0]
    zs, ld_inv = f64.inverse(y16)
    y_back, ld_fwd = f64.forward(zs)
rt = float((y_back - y16).abs().max()); cancel = float((ld_inv + ld_fwd).abs().max())
assert rt < 1e-9 and cancel < 1e-9, (rt, cancel)
print(f"Checkpoint 2.2 passed: round trip {rt:.1e}, log-det cancellation {cancel:.1e}, "
      f"untrained log p_X mean {float(lp.mean()):.1f} nats.")

### TODO 2.3 — The per-level bookkeeping table

`flow.bookkeeping(y)` returns one row per level with per-example tensors:
`coupling_logdet` (the sum of the level's coupling log-dets) and `base_logprob`
($\log p_0$ of the factor that level emits). Reconcile the rows with `log_prob`
**per example** — to float32 rounding, which on terms of size $10^{3}$ is about
$10^{-3}$ — then print the table with batch means.

| level | active shape | coupling log-det | factored latent shape | base log-prob |
|---|---|---:|---|---:|

The table must add up exactly. If it does not, some term was counted twice or not at
all — and there is no squeeze row, because a squeeze contributes nothing.

In [ ]:
with torch.no_grad():
    y16 = logit.inverse(x_test[:16])[0]
    rows = flow_live.bookkeeping(y16)
    lp_y = flow_live.log_prob(y16)
# TODO 2.3: total = sum over rows of (coupling_logdet + base_logprob), per example;
#   assert max |total - lp_y| < 1e-2 (float32 rounding on numbers of size 1e3);
#   then print one line per row with the batch means, and a total line.
raise NotImplementedError

### TODO 2.4 — The live run: pipeline valid, held-out trend downward

The training loss is exact maximum likelihood on the dequantized data, with the
sampling subscripts written out:
$$
\mathcal{L}(\theta) = -\,\mathbb{E}_{k \sim q,\; u \sim \mathrm{Unif}([0,1)^D)}
\Big[\log p_Y^{\theta}\big(T_\lambda(\tfrac{k+u}{256})\big) + \log\lvert\det J_{T_\lambda}(\tfrac{k+u}{256})\rvert\Big].
$$
The Jacobian term does not depend on $\theta$, so the gradient is the same with or
without it; it is kept so that the logged loss converts to bits per dimension directly.
**Fresh noise at every step** — the fixed bank is for the test report, not for training.

Everything about the loop is the given `Trainer`, as in U1.L1: you write the closure.
`model(k)` receives a raw `uint8` batch and must return $\log p_X$; `NLL` returns
minus its mean. The live configuration is small on purpose — $10{,}000$ training
images, $3$ epochs, about 23 s on CPU — and its criterion is *pipeline valid +
held-out BPD trending downward*, not a number to beat. The reference run measured
3.72 → 3.48 bits per dimension over the three epochs.

In [ ]:
class LogProbModel(nn.Module):
    """The Trainer calls model(k) on raw uint8; return log p_X with FRESH noise."""

    def __init__(self, flow):
        super().__init__()
        self.flow = flow

    def forward(self, k):
        # TODO 2.4a: u = torch.rand(k.shape); return log_prob_x(self.flow, k, u)
        raise NotImplementedError


class NLL(nn.Module):
    """loss_fn(model(k), y) -> minus the mean log-density; y is a placeholder."""

    def forward(self, logp, y):
        # TODO 2.4b
        raise NotImplementedError

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fbgm-2026")


def warmup_cosine(step, total, warmup):
    """Multiplier on the base LR -- the U0.T2 default, stepped PER STEP."""
    if step < warmup:
        return step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * p))


def nats_to_bpd_deq(nll_nats):
    return nll_nats / (D * math.log(2.0)) + 8.0


LIVE = dict(n_train=10_000, n_val=2_000, split_seed=0, batch_size=128, epochs=3,
            lr=3e-4, weight_decay=0.01, warmup_frac=0.05, seed=0,
            levels=2, steps=4, hidden=48, lam=LAM)


def fit_flow(model, cfg, run_name):
    """The lab's training call: given Trainer, course defaults, MLflow per epoch."""
    set_seed(cfg["seed"])
    k_all = load_raw(True)
    perm = torch.randperm(k_all.shape[0], generator=torch.Generator().manual_seed(cfg["split_seed"]))
    k_tr, k_va = k_all[perm[:cfg["n_train"]]], k_all[perm[cfg["n_train"]:cfg["n_train"] + cfg["n_val"]]]
    tr = DataLoader(TensorDataset(k_tr, torch.zeros(len(k_tr))), batch_size=cfg["batch_size"],
                    shuffle=True, drop_last=True, generator=torch.Generator().manual_seed(cfg["seed"]))
    va = DataLoader(TensorDataset(k_va, torch.zeros(len(k_va))), batch_size=500)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total = cfg["epochs"] * len(tr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: warmup_cosine(s, total, int(cfg["warmup_frac"] * total)))
    trainer = Trainer(model, opt, NLL(), DEVICE, ckpt_dir=f"runs/{run_name}",
                      log_every=1, sched=sched, config=cfg)
    t0 = time.perf_counter()
    hist = trainer.fit(tr, va, cfg["epochs"])
    hist["seconds"] = time.perf_counter() - t0
    hist["val_bpd_deq"] = [nats_to_bpd_deq(v) for v in hist["val_loss"]]
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({k: str(v) for k, v in cfg.items()})
        mlflow.set_tags({"session": "U1.L2", "data": "MNIST raw uint8", "seconds": f"{hist['seconds']:.0f}"})
        for ep, v in enumerate(hist["val_bpd_deq"], start=1):
            mlflow.log_metrics({"val_bpd_deq": v}, step=ep)
    return hist


hist_live = fit_flow(LogProbModel(flow_live), LIVE, "u1l2-live")
print(f"{hist_live['seconds']:.0f} s on CPU; val bits/dim per epoch: "
      + ", ".join(f"{v:.3f}" for v in hist_live["val_bpd_deq"]))

**Checkpoint 2.4** — the held-out BPD trends downward, and the model is still a
bijection whose table reconciles.

In [ ]:
v = hist_live["val_bpd_deq"]
assert v[-1] < v[0], "the held-out BPD did not trend downward"
flow_live.eval()
with torch.no_grad():
    rows = flow_live.bookkeeping(y16)
    gap = float((sum(r["coupling_logdet"] + r["base_logprob"] for r in rows) - flow_live.log_prob(y16)).abs().max())
assert gap < 1e-2
print(f"Checkpoint 2.4 passed: {v[0]:.3f} -> {v[-1]:.3f} bits/dim; table still reconciles ({gap:.1e}).")

### The reference checkpoint, and its record

Live training demonstrates the pipeline, not convergence. Blocks 4 and 5 read numbers
off an **instructor-provided checkpoint**, and a number read off a checkpoint is only
as good as the record that says which code, which data and which settings produced it.
So the record is verified before anything is read: the file's SHA-256 against the
sidecar, the configuration, the seed, the git commit and whether the tree was dirty,
the training device and time, and the protocol constants it was evaluated under.

The reference is a two-level model, 30 epochs on $55{,}000$ training images
(a seeded $55{,}000 / 5{,}000$ split of the training set; the test set was read once,
in block 4, by the script that built the package). It trained on a GPU
(NVIDIA GeForce RTX 4070 Laptop GPU, 730 s) and evaluates here on CPU. Its companion is the
**one-level baseline** of TODO 4.3: the same conditioner, the same total number of
coupling steps, the same budget, and no factor-out.

In [ ]:
from common.multiscale_flow import Logit as _Logit   # noqa: F401  (same class; documents the dependency)

WEIGHTS = ROOT / "common" / "weights"


def load_checkpoint(name):
    """Verify the record, then load. Returns (flow, payload, sidecar)."""
    path = WEIGHTS / f"{name}.pt"
    side = json.loads((WEIGHTS / f"{name}.json").read_text())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == side["sha256"], "checkpoint hash mismatch"
    payload = torch.load(path, map_location="cpu", weights_only=False)
    c = payload["config"]
    assert c == side["config"] and payload["seed"] == side["seed"]
    flow = MultiscaleFlow((1, 28, 28), levels=c["levels"], steps=c["steps"], hidden=c["hidden"], s_cap=c["s_cap"])
    flow.load_state_dict(payload["state_dict"])
    flow.eval()
    return flow, payload, side


flow_ref, pay_ref, side_ref = load_checkpoint("u1l2-multiscale")
print(f"config {pay_ref['config']}  seed {pay_ref['seed']}  params {pay_ref['n_params']:,}")
print(f"commit {pay_ref['git_commit'][:12]}  dirty={pay_ref['git_dirty']}  "
      f"{pay_ref['device']} {pay_ref['seconds']:.0f} s  selection: {pay_ref['selection']}")
print("protocol constants stored with the checkpoint:", {k: pay_ref["protocol"][k] for k in ("N", "D", "K_deq", "deq_seed", "lam")})
assert pay_ref["protocol"]["deq_seed"] == DEQ_SEED and pay_ref["protocol"]["lam"] == LAM

## Block 3 — What the library supplies, and what the course shell supplies

Zuko is the course's reference implementation for discrete flows, and it is used here
for exactly what it supplies: the coupling arithmetic, transform composition, and a
transform direction to translate. It does **not** supply a squeeze, a factor-out, or a
convolutional conditioner — the multi-scale shell is course code, and the worksheet
below asks you to say so line by line. The anchors were generated from **zuko
1.5.0**, the version pinned in the course environment.

Three cross-checks are executed, not asserted in prose.

**3.1 The logit layer.** `zuko.transforms.SigmoidTransform` — a `torch.distributions` class Zuko
re-exports, as the anchor shows (`torch/distributions/transforms.py:626-648`) — is
$y \mapsto \sigma(y)$ with `.inv` the logit (`torch/distributions/transforms.py:642-645`) and a
log-Jacobian for the sigmoid direction (`torch/distributions/transforms.py:647-648`). Our `Logit.inverse` is
the *opposite* direction, after the affine squeeze into $[\lambda, 1-\lambda]$: so its
log-determinant is **minus** Zuko's, plus $D\log(1-2\lambda)$.

In [ ]:
import zuko
import zuko.transforms as zt

print("zuko", zuko.__version__)
xd = x_test[:8].double()
a = LAM + (1 - 2 * LAM) * xd
sig = zt.SigmoidTransform()
y_lib = sig.inv(a)
ld_lib = -sig.log_abs_det_jacobian(y_lib, a).flatten(1).sum(1) + D * math.log(1 - 2 * LAM)
y_ours, ld_ours = logit.inverse(xd)
print(f"y: |zuko - course| = {float((y_lib - y_ours).abs().max()):.1e}   "
      f"log-det: |zuko - course| = {float((ld_lib - ld_ours).abs().max()):.1e}")
assert float((ld_lib - ld_ours).abs().max()) < 1e-8

**3.2 The coupling arithmetic.** `CouplingTransform(meta, mask)`
(`zuko/transforms.py:1001-1064`) is the skeleton of U1.L1's layer on a *vector*: `_call`
(`zuko/transforms.py:1041-1045`) splits by the mask, calls `meta(x_a)` to get a transform, and
applies it to `x_b`. Plug the **course conv conditioner** in as `meta` — reshaping the
flat vector back to $2 \times 14 \times 14$ — and the library reproduces
`ImageAffineCoupling.inverse` to float precision, log-determinant
(`zuko/transforms.py:1053-1057`) included. Zuko's `_call` is the density direction, so `meta`
returns the *inverse* affine map, $x_B = (y_B - t)\,e^{-s}$.

In [ ]:
set_seed(0)
layer = ImageAffineCoupling(4, [True, True, False, False], hidden=16).double()
h = sq.squeeze(logit.inverse(x_test[:8])[0]).double()          # (8, 4, 14, 14)
mask_vec = torch.zeros(4, 14, 14, dtype=torch.bool); mask_vec[:2] = True


def meta(a_flat):
    s, t = layer._heads(a_flat.view(-1, 2, 14, 14))
    return zt.AffineTransform(loc=(-t * torch.exp(-s)).flatten(1), scale=torch.exp(-s).flatten(1), event_dim=1)


ct = zt.CouplingTransform(meta, mask_vec.flatten())
with torch.no_grad():
    z_lib = ct(h.flatten(1)); ladj_lib = ct.log_abs_det_jacobian(h.flatten(1), z_lib)
    z_ours, ld_ours = layer.inverse(h)
print(f"z: |zuko - course| = {float((z_lib.view_as(h) - z_ours).abs().max()):.1e}   "
      f"log-det: |zuko - course| = {float((ladj_lib - ld_ours).abs().max()):.1e}")
assert float((ladj_lib - ld_ours).abs().max()) < 1e-8

**3.3 The shell.** What is *not* there: no squeeze, no factor-out.
`ReshapeTransform` (torch's, re-exported; `torch/distributions/transforms.py:481-552`) is a plain reshape with log-det $0$
— the *fact* about a squeeze, without the sub-position permutation — and nothing in
the library partitions a state into latent factors.

In [ ]:
names = [n for n in dir(zt) + dir(zuko.flows) if "queeze" in n.lower() or "factor" in n.lower()]
print("squeeze / factor-out in zuko:", names or "none")
assert not names
print("ReshapeTransform present:", hasattr(zt, "ReshapeTransform"))

### TODO 3.4 — The worksheet

For each line of today's pipeline, say which of three things it is: **model
mathematics** (a statement of the change-of-variables theorem or of the observation
model), **image-scale orchestration** (bookkeeping that exists because $d = 784$ is an
image), or **library API convention** (a choice of direction, name or packaging that
another library makes differently). Where Zuko has the object, name it; `meta` is at
`zuko/flows/coupling.py:128-136`.

| Line of the pipeline | Which of the three? | Zuko object, if any |
|---|---|---|
| $x = (k+u)/256$ | | |
| $y = T_\lambda(x)$ and its log-Jacobian | | |
| $y_B = x_B e^{s} + t$, log-det $= \sum s$ | | |
| the conv conditioner $s, t = c_\theta(x_A)$ | | |
| squeeze $1 \times 28 \times 28 \to 4 \times 14 \times 14$ | | |
| factor-out of channels $\{0,1\}$ as $z_1$ | | |
| $\log p_Y = \sum_l \log p_0(z_l) + \sum \log\lvert\det J\rvert$ | | |
| `forward` is $\psi$ (source $\to$ data); Zuko's `_call` is $\psi^{-1}$ | | |
| `bpd(log_prob_x, 784) + 8` | | |

Write your answers in the cell below.

**Your worksheet (TODO 3.4):**

## Block 4 — Evaluate the checkpoint under the frozen protocol

### TODO 4.1 — $\log p_X$ for the fixed test bank, Jacobian included

The whole test set, the bank of block 1, in chunks of $500$, under `no_grad`. On a
laptop core this takes about 14 s. The result is a vector of $10{,}000$
per-example log-densities in nats — the object the harness takes.

In [ ]:
def log_prob_x_all(flow, k, u, chunk=500):
    """(N,1,28,28) uint8, (N,1,28,28) noise -> (N,) log p_X in nats, float64."""
    out = []
    with torch.no_grad():
        for i in range(0, k.shape[0], chunk):
            # TODO 4.1: append log_prob_x(flow, k[i:i+chunk], u[i:i+chunk])
            raise NotImplementedError
    return torch.cat(out).double()


t0 = time.perf_counter()
lp_ref = log_prob_x_all(flow_ref, k_test, u_test)
print(f"{lp_ref.shape[0]} images in {time.perf_counter() - t0:.1f} s; mean log p_X = {float(lp_ref.mean()):.1f} nats")

**Checkpoint 4.1** — $N = 10{,}000$, all finite, and the mean agrees with the reference
run's (3640.9 nats) to CPU rounding.

In [ ]:
assert lp_ref.shape == (10_000,) and bool(torch.isfinite(lp_ref).all())
assert abs(float(lp_ref.mean()) - (3640.9)) < 0.5, "the mean log-density does not match the reference run"
print("Checkpoint 4.1 passed.")

### TODO 4.2 — The reported number, and its record

Call the protocol, verify it against the manual arithmetic on a fixed batch of $16$,
and **log the report** — dataset and split, $N$, dequantization seed and `K_deq`,
$\lambda$, checkpoint hash and config — because a number without its protocol is not a
result. The reference run reports **1.300 bits per dimension** (continuous term
-6.700, plus $8$); on the first $16$ test images alone it is 1.2603.

In [ ]:
# TODO 4.2a: bpd_cont = bpd(lp_ref.numpy(), D); bpd_deq = bpd_cont + 8 (or bpd_protocol).
bpd_cont, bpd_deq = None, None
# TODO 4.2b: on the first 16 images, compute log p_Y, the logit log-det and log p_X
#   separately (flow_ref.log_prob, logit.inverse), and check that
#   bpd_manual(log p_X) == bpd_protocol(log p_X) to 1e-12.
raise NotImplementedError

In [ ]:
report = {"dataset": "MNIST", "split": "test", "N": int(lp_ref.shape[0]), "D": D,
          "K_deq": 1, "deq_seed": DEQ_SEED, "lam": LAM, "checkpoint": "u1l2-multiscale",
          "checkpoint_sha256": side_ref["sha256"], "config": json.dumps(pay_ref["config"]),
          "bpd_cont": float(bpd_cont), "bpd_deq": float(bpd_deq)}
with mlflow.start_run(run_name="u1l2-protocol-multiscale"):
    mlflow.set_tags({k: str(v) for k, v in report.items() if k not in ("bpd_cont", "bpd_deq")})
    mlflow.log_metrics({"bpd_cont": report["bpd_cont"], "bpd_deq": report["bpd_deq"]})
print(json.dumps(report, indent=1))
assert abs(bpd_deq - 1.300) < 2e-3, "the reported number does not match the reference run"
print("Checkpoint 4.2 passed: bpd_deq =", f"{bpd_deq:.4f}")

### TODO 4.3 — A course-internal baseline, under exactly the same protocol

Load `u1l2-onelevel` — one level, sixteen coupling steps at $14 \times 14$, the same
conditioner width, the same budget, and no factor-out — and evaluate it with the **same
bank, the same $\lambda$, the same $N$**. Then compare. This is the quantitative
comparison the protocol allows you to interpret directly: the two numbers were produced
by one measurement procedure, so their difference is a statement about the models.

The reference run measured 1.286 (one level) against 1.300 (two levels) bits
per dimension, a gap of 0.014 in favour of the one-level model; a paired bootstrap
over the shared test images (the same $k$, the same $u$; one-level minus two-level) puts the
95% interval at $[-0.015, -0.014]$. The gap has a price: one density evaluation costs
90.3 MFLOP per image for the one-level model against 60.5 for the
two-level one (a factor of 1.49). "The same number of coupling steps" is not
"the same compute" when half of one model's steps run at $7 \times 7$.

In [ ]:
flow_base, pay_base, side_base = load_checkpoint("u1l2-onelevel")
print(f"baseline: config {pay_base['config']}  params {pay_base['n_params']:,}  {pay_base['seconds']:.0f} s")
assert pay_base["protocol"] == pay_ref["protocol"], "the two checkpoints must carry identical protocol constants"
# TODO 4.3: lp_base = log_prob_x_all(flow_base, k_test, u_test); bpd_base = bpd_protocol(lp_base.numpy());
#   then the paired bootstrap of the per-image difference (given below).
raise NotImplementedError

In [ ]:
d = -(lp_base - lp_ref).numpy() / (D * math.log(2.0))   # per-image bits/dim, baseline minus reference
rng = np.random.default_rng(0)
boots = np.array([d[rng.integers(0, d.size, d.size)].mean() for _ in range(2000)])
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"two levels: {bpd_deq:.4f}   one level: {bpd_base:.4f}   gap: {bpd_base - bpd_deq:+.4f} bits/dim   "
      f"paired bootstrap 95%: [{lo:+.4f}, {hi:+.4f}]")
assert abs(bpd_base - 1.286) < 2e-3
print("Checkpoint 4.3 passed: both models under one protocol.")

### Literature numbers are context, not a leaderboard

[Papamakarios, Pavlakou & Murray (2017)](https://arxiv.org/abs/1705.07057), Table 7,
report MNIST in bits per pixel for fully connected flows: Real NVP with 5 coupling
layers **1.93 ± 0.01**, MAF (5) **1.89 ± 0.01**, MADE MoG **1.41 ± 0.01**. Their
appendix E.2 converts from nats in logit space (their Table 2) with the same
$\lambda = 10^{-6}$ and the same division by $256$ — which is why the course uses this
form of the transform. Glow reports no MNIST number at all; its 3.35 bits/dim and
RealNVP's 3.49 are CIFAR-10.

**Exercise (in words).** Before you place today's 1.300 next to their 1.93, list
every mismatch of protocol you can find between their evaluation and the frozen
`bpd-protocol` — the dequantization draw, the split, the model class, the meaning of
the $\pm$. Then decide which of the two comparisons on this page you are allowed to
interpret as a statement about architectures.

**Your list of mismatches:**

## Block 5 — Samples, temperature, and an honest inspection

### TODO 5.1 — Sample at $\tau \in \{0.7, 1.0\}$

`flow.sample(n, tau)` draws $z_l = \tau\,\varepsilon_l$ with $\varepsilon_l \sim
\mathcal{N}(0, I)$ per factor, runs the generative direction, and returns $y$ in logit
space; `logit.forward` takes it back to $[0, 1]$. At $\tau = 1$ these are samples from
the model whose likelihood you just reported. At $\tau = 0.7$ they are **not**: they are
draws from a modified base $\mathcal{N}(0, \tau^2 I)$ pushed through the same map — a
sampling heuristic, and the panel is labelled as one. Use one fixed noise bank for both
so that the two panels differ by $\tau$ only.

In [ ]:
def sample_x(flow, n, tau, seed=11):
    """n images in [0,1], from a fixed noise bank scaled by tau."""
    g = torch.Generator().manual_seed(seed)
    # TODO 5.1: y = flow.sample(n, tau, generator=g); x = logit.forward(y)[0]; clamp to [0, 1].
    raise NotImplementedError


def grid(x, nrow=8):
    n, _, h, w = x.shape
    canvas = np.ones(((n // nrow) * h, nrow * w), dtype=np.float32)
    for i in range(n):
        r, c = divmod(i, nrow)
        canvas[r * h:(r + 1) * h, c * w:(c + 1) * w] = x[i, 0].numpy()
    return canvas


fig, axes = plt.subplots(1, 2, figsize=(9, 4.8))
for ax, tau, title in zip(axes, (1.0, 0.7), ("tau = 1.0: samples from the model",
                                             "tau = 0.7: a heuristic, modified base N(0, 0.49 I)")):
    ax.imshow(grid(sample_x(flow_ref, 64, tau)), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

**Checkpoint 5.1** — shapes and range; and one measured fact about the heuristic: the
model assigns its own $\tau = 0.7$ draws a *different* density than its $\tau = 1$
draws (the reference run measured -0.451 bits/dim, $\tau{=}0.7$ minus
$\tau{=}1$). Draws that the model scores differently from its own samples are, by that
measurement, not its samples.

In [ ]:
xs1, xs7 = sample_x(flow_ref, 64, 1.0), sample_x(flow_ref, 64, 0.7)
assert xs1.shape == (64, 1, 28, 28) and float(xs1.min()) >= 0 and float(xs1.max()) <= 1
with torch.no_grad():
    def bpd_of(x):
        y, ld = logit.inverse(x.clamp(1e-6, 1 - 1e-6))
        return bpd_protocol((flow_ref.log_prob(y) + ld).double().numpy())
    b1, b7 = bpd_of(xs1), bpd_of(xs7)
print(f"bits/dim the model assigns to its own draws: tau=1.0 {b1:.3f}   tau=0.7 {b7:.3f}   gap {b7 - b1:+.3f}")
assert b1 != b7
print("Checkpoint 5.1 passed.")

### TODO 5.2 — Likelihood and samples are different observables

Draw the one-level baseline at $\tau = 1$ beside the reference at $\tau = 1$, and put
each model's bits per dimension in the title. Then answer, in the cell after the figure,
and answer only what you can see and what you measured:

> **Does the model with the better likelihood produce the better-looking samples? What
> would you need to say more than "looks"?**

The narrow lesson: an exact likelihood and the visual quality of samples are different
observables of the same model, and [Theis, van den Oord & Bethge
(2016)](https://arxiv.org/abs/1511.01844), §3.2, give constructions where each is good
while the other is poor. A *sample* metric is the payoff of U3.L2; it is not introduced
here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.8))
# TODO 5.2: reference at tau = 1 on axes[0] with title f"two levels: {bpd_deq:.3f} bits/dim";
#   baseline at tau = 1 on axes[1] with title f"one level: {bpd_base:.3f} bits/dim".
raise NotImplementedError

**Your observation (TODO 5.2, in words):**

## Block 6 — Close: what an exact likelihood buys, and costs, at image scale

The two-costs table, version 2 (U1.T2 §8), with one annotation per row for today.

| Family | Density / likelihood | Sampling | At image scale |
|---|---|---|---|
| NICE / RealNVP | exact: analytic inverse, triangular log-det | direct | conditioners become convolutions; the stack needs squeeze and factor-out to stay tractable in parameters and memory (RealNVP §3.6) |
| Glow | exact | direct | actnorm and $1 \times 1$ convolutions slot into each step; the bookkeeping of U1.T2 §2 adds two sums per step |
| MAF / IAF | one direction parallel, the other sequential in $d$ | | sequential in $784$ is the cost that made the fully connected MNIST flows of Table 7 stop at 5–10 layers |
| Residual flow | direct map; log-det a stochastic series | iterative inverse | the series and the estimator scale with $d$; U2.T2 |

The asymptotics of the "cheap" coupling row did not change today. What changed is the
constant in front: 275,040 parameters, 60.5 MFLOP per density evaluation
and 730 GPU-seconds for a number on MNIST, and every bit of it exact. TODO 4.3
is the multi-scale trade measured: at matched steps and epochs, factoring out half the
state after eight steps cost 0.014 bits per dimension and bought a third of the
compute.

Two facts you now own:

1. **An exact likelihood is only meaningful after the observation and dequantization
   convention is specified.** The same checkpoint reports a different number under a
   different $\lambda$, a different bank, or without the $+8$ — and it reports
   1.300 under `bpd-protocol`.
2. **A multi-scale architecture changes bookkeeping and compute, not the
   change-of-variables theorem.** The table of TODO 2.3 is the theorem, summed over a
   partition.

And the question U1.T2 left on the board, now with a number attached to the answer's
cost: a two-level stack accumulates its log-determinant over sixteen coupling steps.
**What does that accumulated log-determinant become when the depth becomes
continuous?** — U2.

### TODO 6.1 — For PS1.4(d): the bookkeeping on frozen outputs (at home, or if this session was cut)

`labs/ps1-4d-bundle.npz` holds sixteen raw `uint8` test images, their fixed
dequantization noise, and the reference checkpoint's **frozen $\log p_Y$** at the
corresponding $y$. No model, no GPU, no `torch` needed: reconstruct, with `numpy`
only, $x \to y$, the preprocessing Jacobian, $\log p_X$, the continuous bits per
dimension, and the $+8$, and compare with the value recorded in the bundle
(1.2603).

In [ ]:
bundle = np.load(ROOT / "labs" / "ps1-4d-bundle.npz")
print({k: (bundle[k].shape if bundle[k].ndim else bundle[k].item()) for k in bundle.files})
k16, u16, lpy16_frozen = bundle["k_uint8"].astype(np.float64), bundle["u"].astype(np.float64), bundle["log_prob_y"]
# TODO 6.1 (numpy only): x = (k + u)/256; a = lam + (1-2 lam) x; ld = sum log(1-2lam) - log a - log(1-a);
#   log p_X = log p_Y + ld; bpd_cont = -mean/(784 log 2); bpd_deq = bpd_cont + 8. Compare with the bundle.
raise NotImplementedError